# RG3 gas: seed sensitivity of the candidate innovation confirmation
Frozen baseline RF. Same fixed barrel5 current rise > internal normal95% and expert > internal normal99.5%.
Sensitivity seeds42/7/101; seed42 reused from42, additional7/101 refit with identical positive clusters. No seed winner selection; three-seed mean/median evaluated.
Outer block always excluded. Repeated-data sensitivity is NOT independent new production validation.
Thresholds for each seed/ensemble are refit only from corresponding train-inner normal scores.

In [ ]:
from pathlib import Path
import json
import numpy as np,pandas as pd
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
nb=json.loads((root/'notebooks/22_rg3_nested_signal_ensemble.ipynb').read_text(encoding='utf-8'))
exec(''.join(nb['cells'][1]['source']).split('scores=[];thresholds=[];foldlogs=[]')[0])
original=pd.read_csv(tables/'rg3_nested_ensemble_predictions.csv',parse_dates=['TimeStamp'])
original=original.loc[original.model.eq('max_union'),['scheme','TimeStamp','predicted_train_fpr1']]
oldouter=pd.read_csv(tables/'rg3_rank_fusion_predictions.csv',parse_dates=['TimeStamp'])
oldouter=oldouter.loc[oldouter.model.eq('baseline_rank'),['scheme','TimeStamp','block','actual','raw_expert_score']]
oldinner=pd.read_csv(tables/'rg3_rank_fusion_inner_scores.csv',parse_dates=['TimeStamp'])
memo={}
def expert(ref,q):
    key=(tuple(ref),tuple(q))
    if key in memo:return memo[key]
    assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
    yy=y[ref];pos=yy==1;normal=yy==0
    if len(np.unique(yy))<2:return np.zeros((len(q),2))
    rr=np.full((len(ref),4),np.nan)
    for b in groups.iloc[ref].unique():
        mask=groups.iloc[ref].eq(b).to_numpy()
        if (~mask).any() and (y[ref[~mask]]==0).any():rr[mask]=res_features(ref[~mask],ref[mask])
    views=[(d.iloc[ref][stdcols].to_numpy(float),d.iloc[q][stdcols].to_numpy(float)),(rr,res_features(ref,q))]
    scores=[[],[]]
    for tr,te in views:
        imp=SimpleImputer(strategy='median',keep_empty_features=True)
        tr=imp.fit_transform(tr);te=imp.transform(te)
        sc=RobustScaler().fit(tr[normal]);z=sc.transform(tr)
        k=min(2,len(np.unique(z[pos],axis=0)))
        labels=KMeans(n_clusters=k,n_init=10,random_state=42).fit_predict(z[pos]) if k>1 else np.zeros(pos.sum(),int)
        ids=np.flatnonzero(pos)
        for s,seed in enumerate([7,101]):
            out=[]
            for mode in np.unique(labels):
                chosen=np.r_[np.flatnonzero(normal),ids[labels==mode]]
                m=RandomForestClassifier(n_estimators=100,max_depth=8,min_samples_leaf=3,class_weight='balanced_subsample',random_state=seed,n_jobs=2)
                out.append(m.fit(tr[chosen],yy[chosen]).predict_proba(te)[:,1])
            scores[s].append(np.mean(out,axis=0))
    memo[key]=np.column_stack([np.max(v,axis=0) for v in scores])
    return memo[key]
records=[];audit=[];rawrecords=[]
for scheme in ['leave_one_block_out','past_blocks_only']:
    for j,b in enumerate(blocks):
        q=np.flatnonzero(groups.eq(b));ref=np.flatnonzero(~groups.eq(b)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
        if len(np.unique(y[ref]))<2:continue
        out_extra=expert(ref,q);inner_extra=np.full((len(ref),2),np.nan)
        for ib in groups.iloc[ref].unique():
            mask=groups.iloc[ref].eq(ib).to_numpy()
            if (~mask).any():inner_extra[mask]=expert(ref[~mask],ref[mask])
        oc=oldouter.loc[oldouter.scheme.eq(scheme)&oldouter.block.eq(b)].set_index('TimeStamp').loc[d.TimeStamp.iloc[q]]
        ic=oldinner.loc[oldinner.scheme.eq(scheme)&oldinner.heldout_block.eq(b)].set_index('TimeStamp').loc[d.TimeStamp.iloc[ref]]
        outer=np.column_stack([oc.raw_expert_score.to_numpy(),out_extra])
        inner=np.column_stack([ic.raw_expert_score.to_numpy(),inner_extra])
        normal=(y[ref]==0)&np.isfinite(inner).all(axis=1)
        base=original.loc[original.scheme.eq(scheme)].set_index('TimeStamp').loc[d.TimeStamp.iloc[q],'predicted_train_fpr1'].to_numpy(bool)
        # Fixed confirmation protocol from46, no winner selection among seeds.
        delta=d.Barrel_Temperature_5__delta1.to_numpy(float)
        nv=delta[ref][normal&np.isfinite(delta[ref])]
        delta_cut=float(np.quantile(nv,.95,method='higher'))
        scores={'seed42':(inner[:,0],outer[:,0]),'seed7':(inner[:,1],outer[:,1]),'seed101':(inner[:,2],outer[:,2]),'three_seed_mean':(inner.mean(axis=1),outer.mean(axis=1)),'three_seed_median':(np.median(inner,axis=1),np.median(outer,axis=1))}
        for name,(cal,score) in scores.items():
            cut=float(np.quantile(cal[normal],.995,method='higher'))
            add=(score>cut)&(delta[q]>delta_cut)
            final=base|add
            for k,i in enumerate(q):records.append(dict(scheme=scheme,model=name,TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),prediction=int(final[k]),baseline_prediction=int(base[k]),expert_score=float(score[k]),expert_threshold=cut,current_barrel5_delta=float(delta[i]),delta_threshold=delta_cut))
        audit.append(dict(scheme=scheme,block=b,train=len(ref),train_gas=int(y[ref].sum()),test=len(q),test_gas=int(y[q].sum()),train_test_overlap=0))
        for k,i in enumerate(q):rawrecords.append(dict(scheme=scheme,TimeStamp=d.TimeStamp.iloc[i],actual=int(y[i]),seed42=float(outer[k,0]),seed7=float(outer[k,1]),seed101=float(outer[k,2])))
        print('Finished innovation seed sensitivity',scheme,b,flush=True)
pp=pd.DataFrame(records);rows=[]
for (scheme,name),g in pp.groupby(['scheme','model'],sort=False):
    tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
    rows.append(dict(scheme=scheme,model=name,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn)))
metrics=pd.DataFrame(rows)
pp.to_csv(tables/'rg3_innovation_seed_sensitivity_predictions.csv',index=False,encoding='utf-8-sig')
metrics.to_csv(tables/'rg3_innovation_seed_sensitivity_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(audit).to_csv(tables/'rg3_innovation_seed_sensitivity_fold_audit.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(rawrecords).to_csv(tables/'rg3_innovation_seed_sensitivity_raw_scores.csv',index=False,encoding='utf-8-sig')
assert (pp.prediction>=pp.baseline_prediction).all()
print(metrics.to_string(index=False))
print(pp.loc[pp.actual.eq(1)&pp.prediction.eq(1)&pp.baseline_prediction.eq(0)].to_string(index=False))


## Executed results
Seed42 and101 whole5/6, seed7 whole4/6. Prespecified uniform three-seed mean and median whole TP5 FP6 FN12 TN560 F1=0.357; forward TP3 FP4 FN10 TN423 F1=0.300. New case07:21:20; same last block across schemes, not two independent validations.
See reports/rg3_current_innovation_confirmation.md and reports/rg3_remaining12_cold_signature_review.md.